# §57 — EDA of the cut DoTA train corpus (`DoTA_train_window`)

Rule: `core/docs/final/KIM_RUN.md` **§57**. Upload branch `v_final_las_vad` to `Thesis-Final/kat-vad` first. **CPU only** (no GPU needed), about 2-3 min. Dataset properties only: no model, nothing gates, no arm follows.

The windows are rebuilt from DoTA's annotation zip and the frozen train map with the planner that wrote the corpus, then **gated** window by window against `Thesis-Final/data/DoTA_train_window` (§55).

| step | what |
|---|---|
| 1-2 | mount, paths, code check |
| 3 | annotation zip (sha256 checked against the frozen map) |
| 4 | tests of the touched code — **raises if red** |
| 5 | EDA + gate + figures -> `Thesis-Final/outputs/dota_eda/REPORTS/`; **paste `dota_eda_report.md` to Claude** |

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import hashlib
import os
import shutil
import subprocess
import sys
import urllib.request
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only
DRIVE_F  = '/content/drive/MyDrive/Thesis-Final'  # code, the §55 corpus; every output of this notebook
os.environ['PROJECT_ROOT']       = DRIVE_F
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE_F}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_F}/outputs'

REPO = Path(f'{DRIVE_F}/kat-vad')
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v_final_las_vad there'
assert '## 57. ' in (REPO / 'core/docs/final/KIM_RUN.md').read_text(), 'KIM_RUN.md has no §57 -- re-upload kat-vad/'
assert (REPO / 'core/tools/dota_eda.py').is_file(), 'no core/tools/dota_eda.py -- re-upload kat-vad/'
assert 'def plan(' in (REPO / 'core/tools/dota_train_build.py').read_text(), 'dota_train_build has no plan() -- re-upload kat-vad/'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
DATA    = Path(DRIVE_F) / 'data' / constants.DOTA_TRAIN_CORPUS_DIRNAME
REPORTS = constants.OUTPUT_ROOT / constants.DOTA_EDA_OUTPUT_DIRNAME / 'REPORTS'
LOCAL   = Path('/content/dota_eda')
LOCAL.mkdir(parents=True, exist_ok=True)
for f in (constants.LABELS_TRAIN_FILENAME, constants.META_FILENAME):
    assert (DATA / f).is_file(), f'no {DATA / f}: run k5dfidota.ipynb (§55) first'
print(f'corpus {DATA} | reports -> {REPORTS}')


def run(cmd):
    """Stream a child's output; raise with its tail on failure."""
    tail = []
    env = dict(os.environ, MPLBACKEND='Agg')
    with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=env, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
        for line in proc.stdout:
            tail = (tail + [line])[-40:]
            print(line, end='')
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}')

In [ ]:
import json  # noqa: E402

ANN = LOCAL / 'DoTA_annotations.zip'              # DoTA's official annotation zip (10 MB)
if not ANN.exists():
    urllib.request.urlretrieve('https://github.com/MoonBlvd/Detection-of-Traffic-Anomaly/raw/master/dataset/DoTA_annotations.zip', ANN)
frozen = json.loads((constants.DOTA_TRAIN_SPLITS_DIR / constants.DOTA_TRAIN_MAP_FILENAME).read_text())
assert hashlib.sha256(ANN.read_bytes()).hexdigest() == frozen['sources']['annotations_sha256'], 'annotation zip is not the one the map was built on'
print('annotations OK')

## 4. Tests (CPU) — **raises if anything fails: stop and paste the output**

In [ ]:
run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', 'core/tests/test_dota_eda.py',
     'core/tests/test_dota_train_corpus.py', 'core/tests/test_nexar_eda.py'])

## 5. EDA + gate + figures -> **paste `REPORTS/dota_eda_report.md` to Claude**

In [ ]:
from IPython.display import Image as Show, Markdown, display  # noqa: E402

out = LOCAL / 'report'
shutil.rmtree(out, ignore_errors=True)
run([sys.executable, '-m', 'core.tools.dota_eda', '--annotations', ANN, '--data-dir', DATA, '--output-dir', out, '--plots'])
REPORTS.mkdir(parents=True, exist_ok=True)
shutil.copytree(out, REPORTS, dirs_exist_ok=True)
report = json.loads((REPORTS / f'{constants.DOTA_EDA_REPORT_STEM}.json').read_text())
assert report['gate'] and report['gate']['equal'] and report['gate']['windows_checked'] == 12937, report['gate']
display(Markdown((REPORTS / f'{constants.DOTA_EDA_REPORT_STEM}.md').read_text()))
for name in report['figures']:
    print(name)
    display(Show(filename=str(REPORTS / constants.NEXAR_EDA_FIGURES_DIRNAME / name)))
drive.flush_and_unmount()          # C10: every pending Drive write lands before the VM goes
print('Drive flushed -- paste REPORTS/dota_eda_report.md to Claude')